# LambdaMART ranking with query groups

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/supervised/lambdamart_ranking.ipynb)

This notebook runs the complete CPU experiment. Results depend on package versions and hardware. No GPU or credentials are required.

In [ ]:
%pip -q install "numpy>=2,<3" "scipy>=1.14,<2" "scikit-learn>=1.6,<2" "xgboost==3.4.1" "matplotlib>=3.9,<4"

## Data and evaluation

The public LightGBM sample contains 3005 documents in 201 query groups. We download two files from an immutable source revision and verify SHA-256 hashes. LightGBM is MIT licensed; the source files stay at their original host. See the [repository licence](https://github.com/lightgbm-org/LightGBM/blob/3ee03c26ba8c9e8ddc05fa8687717cae8a7d61d9/LICENSE).

Every split holds out 41 entire query groups. NDCG@10 uses exponential gain, averages ties and gives a query with no relevant documents score one. That last convention is explicit, and can be changed for a sensitivity check.

In [ ]:
import os, tempfile, time, json
for name in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:
    os.environ[name] = '4'
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path(tempfile.mkdtemp(prefix='ml-tutorial-'))
import urllib.request, hashlib
from scipy.special import expit
from sklearn.datasets import load_svmlight_file
from sklearn.metrics import ndcg_score
from xgboost import XGBRanker, XGBRegressor
(ROOT / "rank_results").mkdir()
REV='3ee03c26ba8c9e8ddc05fa8687717cae8a7d61d9'
BASE=f'https://raw.githubusercontent.com/lightgbm-org/LightGBM/{REV}/examples/lambdarank/'
HASHES={'rank.train':'a0c7201c89120879c14a5059e091f441cbf2a29b8aaef363885ccb1a530448df','rank.train.query':'40cb5fef637229311cd30fab13cb7fb633b023e86a87ff04ec55db420143d6af'}
def rank_data(cache=Path('/tmp/batch12-ranking-data')):
    cache.mkdir(exist_ok=True)
    for name,sha in HASHES.items():
        p=cache/name
        if not p.exists():p.write_bytes(urllib.request.urlopen(BASE+name).read())
        assert hashlib.sha256(p.read_bytes()).hexdigest()==sha
    X,y=load_svmlight_file(str(cache/'rank.train'));g=np.loadtxt(cache/'rank.train.query',dtype=int)
    assert g.sum()==X.shape[0]
    return X,y,np.repeat(np.arange(len(g)),g)
def query_ndcg(y,p,q):
    vals=[]
    for group in np.unique(q):
        m=q==group
        vals.append(1.0 if y[m].max()==0 else float(ndcg_score([(2**y[m]-1)], [p[m]],k=10)))
    return np.array(vals)
def ranking():
    start=time.perf_counter();X,y,q=rank_data();res=[];preds={}
    for seed in [11,22,33]:
        rng=np.random.default_rng(seed);groups=rng.permutation(np.unique(q));test=groups[:41]
        trainmask=~np.isin(q,test);testmask=~trainmask
        assert not(set(q[trainmask]) & set(q[testmask]))
        for name,obj in [('Regression','reg:squarederror'),('Pairwise','rank:pairwise'),('LambdaMART','rank:ndcg')]:
            cls=XGBRegressor if name=='Regression' else XGBRanker
            model=cls(objective=obj,n_estimators=100,max_depth=3,learning_rate=.08,tree_method='hist',n_jobs=4,random_state=seed)
            kw={} if name=='Regression' else {'qid':q[trainmask]}
            model.fit(X[trainmask],y[trainmask],**kw)
            pred=model.predict(X[testmask]);scores=query_ndcg(y[testmask],pred,q[testmask])
            res.append({'seed':seed,'method':name,'mean_ndcg10':float(scores.mean()),'query_scores':scores.tolist(),'test_groups':test.tolist()})
            if seed==11:preds[name]=np.array([query_ndcg(y[testmask],model.predict(X[testmask],iteration_range=(0,n)),q[testmask]).mean() for n in range(1,101)])
    # Pairwise loss finite-difference check, and gradient descent on document scores.
    labels=np.array([3,0,2,1]);scores=np.array([-.2,.4,.1,.3]);history=[]
    pairs=[(i,j) for i in range(4) for j in range(4) if labels[i]>labels[j]]
    def loss(s):return sum(np.logaddexp(0,-(s[i]-s[j])) for i,j in pairs)
    def grad(s):
        g=np.zeros_like(s)
        for i,j in pairs:
            v=-expit(-(s[i]-s[j]));g[i]+=v;g[j]-=v
        return g
    eps=1e-6;numeric=np.array([(loss(scores+eps*np.eye(4)[i])-loss(scores-eps*np.eye(4)[i]))/(2*eps) for i in range(4)])
    assert np.max(abs(numeric-grad(scores)))<1e-7
    for _ in range(40):history.append(scores.copy());scores-=.15*grad(scores)
    report={'source_revision':REV,'hashes':HASHES,'rows':X.shape[0],'features':X.shape[1],'groups':len(np.unique(q)),'all_zero_groups':int(sum(np.max(y[q==i])==0 for i in np.unique(q))),'results':res,'gradient_max_error':float(max(abs(numeric-grad(history[0])))),'toy_initial_loss':float(loss(history[0])),'toy_final_loss':float(loss(scores)),'elapsed_seconds':time.perf_counter()-start}
    out=ROOT/'rank_results';(out/'results.json').write_text(json.dumps(report,indent=2));np.savez(out/'arrays.npz',history=history,labels=labels,**preds)
    return report



In [ ]:
report = ranking()
for row in report['results']:
    print(row['seed'], row['method'], round(row['mean_ndcg10'], 4))
print('Gradient check:', report['gradient_max_error'])
print('Runtime:', report['elapsed_seconds'])

## Query groups and metric-aware gradients

For a preferred document pair, the logistic loss is log(1 + exp(-(score_i-score_j))). Its two score derivatives sum to zero. LambdaRank weights these directions by the absolute NDCG change caused by swapping the pair. LambdaMART fits regression trees to the resulting document-level updates. The toy finite-difference check above verifies the unweighted pairwise derivative; XGBoost supplies the complete ranking implementation.

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
for method in ['Regression', 'Pairwise', 'LambdaMART']:
    rows = [r for r in report['results'] if r['method'] == method]
    ax.plot([r['seed'] for r in rows], [r['mean_ndcg10'] for r in rows], 'o-', label=method)
ax.set(xlabel='Group-split seed', ylabel='Mean query NDCG@10')
ax.legend(); plt.show()

## Exercises

1. Change tree depth while holding the query splits fixed. Use another validation group split for model selection.
2. Change k from ten to three and compare per-query differences.
3. Exclude all-zero relevance queries and report the denominator.
4. Add NDCG swap weights to the toy score-gradient calculation.

## References

- [Burges (2010), From RankNet to LambdaRank to LambdaMART](https://www.microsoft.com/en-us/research/wp-content/uploads/2016/02/MSR-TR-2010-82.pdf).
- [XGBoost ranking documentation](https://xgboost.readthedocs.io/en/stable/tutorials/learning_to_rank.html).

## Tested environment
Executed on Python 3.12 with NumPy 2.5.3, SciPy 1.18.1, scikit-learn 1.9.1 and Matplotlib 3.11.2. Ranking additionally used XGBoost 3.4.1. Package versions and floating-point libraries can affect timings and fitted results.
